## Example 1: Read a legacy .xls worksheet built byte by byte

In [ ]:
import importlib.metadata
import struct

import xlrd

rec = lambda rid, d: struct.pack("<HH", rid, len(d)) + d
num = lambda r, c, v: rec(3, struct.pack("<HH", r, c) + bytes(3) + struct.pack("<d", v))
lab = lambda r, c, s: rec(4, struct.pack("<HH", r, c) + bytes(3) + bytes([len(s)]) + s.encode("latin-1"))
xls = lambda cells, nr, nc: (rec(9, struct.pack("<HH", 7, 16)) + rec(0, struct.pack("<HHHH", 0, nr, 0, nc))
                             + b"".join(cells) + rec(10, b""))

# Example 1: two header labels and two data rows, read back with their cell types
assert importlib.metadata.version("xlrd") == "2.0.2"
data = xls([lab(0, 0, "city"), lab(0, 1, "pop"), lab(1, 0, "Oslo"), num(1, 1, 709037.0),
            lab(2, 0, "Bergen"), num(2, 1, 291940.0)], 3, 2)
book = xlrd.open_workbook(file_contents=data, encoding_override="cp1252")
sheet = book.sheet_by_index(0)
assert (book.biff_version, sheet.nrows, sheet.ncols) == (20, 3, 2), (book.biff_version, sheet.nrows, sheet.ncols)
assert sheet.row_values(1) == ["Oslo", 709037.0] and sheet.cell_type(1, 1) == xlrd.XL_CELL_NUMBER
print("Example 1 OK:", [sheet.row_values(i) for i in range(sheet.nrows)])

## Example 2: Excel serial dates in both date systems

In [ ]:
import datetime

import xlrd

# Example 2: Excel stores dates as day counts from 1899-12-30 (datemode 0) or from 1904-01-01 (datemode 1)
assert xlrd.xldate_as_datetime(45000.5, 0) == datetime.datetime(2023, 3, 15, 12, 0)
assert xlrd.xldate_as_tuple(45000.5, 0) == (2023, 3, 15, 12, 0, 0)
assert xlrd.xldate_as_datetime(45000.5 - 1462, 1) == datetime.datetime(2023, 3, 15, 12, 0)
assert xlrd.xldate.xldate_from_date_tuple((2026, 11, 7), 0) == 46333.0
print("Example 2 OK:", xlrd.xldate_as_datetime(46333.25, 0))

## Example 3: Columns, slices and cell objects

In [ ]:
import struct

import xlrd

rec = lambda rid, d: struct.pack("<HH", rid, len(d)) + d
num = lambda r, c, v: rec(3, struct.pack("<HH", r, c) + bytes(3) + struct.pack("<d", v))
lab = lambda r, c, s: rec(4, struct.pack("<HH", r, c) + bytes(3) + bytes([len(s)]) + s.encode("latin-1"))
xls = lambda cells, nr, nc: (rec(9, struct.pack("<HH", 7, 16)) + rec(0, struct.pack("<HHHH", 0, nr, 0, nc))
                             + b"".join(cells) + rec(10, b""))

# Example 3: column access, row slices and the Cell objects that carry a value and a type
cells = [lab(0, 0, "k"), lab(0, 1, "v")] + [x for r in range(1, 6) for x in (lab(r, 0, f"k{r}"), num(r, 1, r * 1.5))]
sheet = xlrd.open_workbook(file_contents=xls(cells, 6, 2), encoding_override="cp1252").sheet_by_index(0)
assert sheet.col_values(1, start_rowx=1) == [1.5, 3.0, 4.5, 6.0, 7.5]
row = sheet.row_slice(3)
assert [c.ctype for c in row] == [xlrd.XL_CELL_TEXT, xlrd.XL_CELL_NUMBER] and row[1].value == 4.5
assert xlrd.cellname(3, 1) == "B4" and xlrd.colname(27) == "AB"
print("Example 3 OK:", sum(sheet.col_values(1, 1)))

## Example 4: pandas.read_excel(engine='xlrd')

In [ ]:
import io
import struct

import pandas as pd

rec = lambda rid, d: struct.pack("<HH", rid, len(d)) + d
num = lambda r, c, v: rec(3, struct.pack("<HH", r, c) + bytes(3) + struct.pack("<d", v))
lab = lambda r, c, s: rec(4, struct.pack("<HH", r, c) + bytes(3) + bytes([len(s)]) + s.encode("latin-1"))
xls = lambda cells, nr, nc: (rec(9, struct.pack("<HH", 7, 16)) + rec(0, struct.pack("<HHHH", 0, nr, 0, nc))
                             + b"".join(cells) + rec(10, b""))

# Example 4: pandas recognises the legacy .xls signature and reads it through xlrd
data = xls([lab(0, 0, "name"), lab(0, 1, "score"), lab(1, 0, "ada"), num(1, 1, 9.5),
            lab(2, 0, "bob"), num(2, 1, 7.0)], 3, 2)
df = pd.read_excel(io.BytesIO(data), engine="xlrd")
assert list(df.columns) == ["name", "score"] and df["score"].sum() == 16.5, df
assert df.loc[0, "name"] == "ada"
print("Example 4 OK:\n", df)

## Example 5: xlrd 2.x reads .xls only and says so for .xlsx

In [ ]:
import io

import xlrd
import xlsxwriter

# Example 5: since 2.0 xlrd refuses .xlsx with a clear error instead of misreading it
buf = io.BytesIO()
with xlsxwriter.Workbook(buf) as wb:
    wb.add_worksheet().write(0, 0, "x")
try:
    xlrd.open_workbook(file_contents=buf.getvalue())
    raise AssertionError("xlrd opened an .xlsx")
except xlrd.biffh.XLRDError as error:
    assert "xlsx" in str(error).lower(), error
    print("Example 5 OK:", error)